In [ ]:
# ==========================================
# CELL 1: INSTALL LIBRARIES (Colab Only)
# ==========================================
!pip install xgboost lightgbm category_encoders -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 6.1 MB/s eta 0:00:00


In [ ]:
# ==========================================
# CELL 2: IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

print("✅ Libraries Loaded")

✅ Libraries Loaded


In [ ]:
# ==========================================
# CELL 3: LOAD TRAIN + TEST DATA
# Upload train.csv and test.csv first
# ==========================================
train_raw = pd.read_csv("/train.csv", parse_dates=["Dates"])
test_raw  = pd.read_csv("/test.csv", parse_dates=["Dates"])

print("Train Shape:", train_raw.shape)
print("Test Shape :", test_raw.shape)

Train Shape: (878049, 9)
Test Shape : (884262, 7)


In [ ]:
# ==========================================
# CELL 4: BASIC CLEANING
# ==========================================
train = train_raw.copy()
test  = test_raw.copy()

for col in ["Category","PdDistrict","DayOfWeek","Address"]:
    if col in train.columns:
        train[col] = train[col].str.strip().str.upper()

for col in ["PdDistrict","DayOfWeek","Address"]:
    test[col] = test[col].str.strip().str.upper()

# Remove train geo outliers
train = train[
    train["X"].between(-122.55, -122.33) &
    train["Y"].between(37.65, 37.85)
]

train.drop_duplicates(inplace=True)
train.reset_index(drop=True, inplace=True)

print("✅ Cleaned Train:", train.shape)
print("✅ Cleaned Test :", test.shape)

✅ Cleaned Train: (875659, 9)
✅ Cleaned Test : (884262, 7)


In [ ]:
# ==========================================
# CELL 5: FEATURE ENGINEERING FUNCTION
# ==========================================
def make_features(df):

    data = df.copy()

    # Time
    data["Hour"] = data["Dates"].dt.hour
    data["Month"] = data["Dates"].dt.month
    data["Year"] = data["Dates"].dt.year
    data["Day"] = data["Dates"].dt.day
    data["WeekdayNum"] = data["Dates"].dt.dayofweek

    # Weekend
    data["IsWeekend"] = data["WeekdayNum"].isin([5,6]).astype(int)

    # Cyclical
    data["HourSin"] = np.sin(2*np.pi*data["Hour"]/24)
    data["HourCos"] = np.cos(2*np.pi*data["Hour"]/24)

    # Address
    data["IsBlock"] = data["Address"].str.contains("BLOCK").astype(int)
    data["IsIntersection"] = data["Address"].str.contains("/").astype(int)

    # Grid
    data["X_round2"] = data["X"].round(2)
    data["Y_round2"] = data["Y"].round(2)

    return data

In [ ]:
# ==========================================
# CELL 6: APPLY FEATURES
# ==========================================
train = make_features(train)
test  = make_features(test)

print("✅ Features Added")
print(train.head(2))

✅ Features Added
                Dates        Category                  Descript  DayOfWeek  \
0 2015-05-13 23:53:00        WARRANTS            WARRANT ARREST  WEDNESDAY   
1 2015-05-13 23:53:00  OTHER OFFENSES  TRAFFIC VIOLATION ARREST  WEDNESDAY   

  PdDistrict      Resolution             Address           X          Y  Hour  \
0   NORTHERN  ARREST, BOOKED  OAK ST / LAGUNA ST -122.425892  37.774599    23   
1   NORTHERN  ARREST, BOOKED  OAK ST / LAGUNA ST -122.425892  37.774599    23   

   ...  Year  Day  WeekdayNum  IsWeekend   HourSin   HourCos  IsBlock  \
0  ...  2015   13           2          0 -0.258819  0.965926        0   
1  ...  2015   13           2          0 -0.258819  0.965926        0   

   IsIntersection  X_round2  Y_round2  
0               1   -122.43     37.77  
1               1   -122.43     37.77  

[2 rows x 21 columns]


In [ ]:
# ==========================================
# CELL 7: LABEL ENCODING
# ==========================================
le_district = LabelEncoder()
le_day      = LabelEncoder()
le_target   = LabelEncoder()

# Fit on combined values for safety
all_districts = pd.concat([train["PdDistrict"], test["PdDistrict"]])
all_days      = pd.concat([train["DayOfWeek"], test["DayOfWeek"]])

le_district.fit(all_districts)
le_day.fit(all_days)

train["District_enc"] = le_district.transform(train["PdDistrict"])
test["District_enc"]  = le_district.transform(test["PdDistrict"])

train["Day_enc"] = le_day.transform(train["DayOfWeek"])
test["Day_enc"]  = le_day.transform(test["DayOfWeek"])

train["Target"] = le_target.fit_transform(train["Category"])

print("✅ Encoding Complete")
print("Classes:", len(le_target.classes_))

✅ Encoding Complete
Classes: 39


In [ ]:
# ==========================================
# CELL 8: FINAL FEATURES
# ==========================================
FEATURES = [
    "Hour","Month","Year","Day","WeekdayNum",
    "IsWeekend",
    "HourSin","HourCos",
    "X","Y",
    "X_round2","Y_round2",
    "IsBlock","IsIntersection",
    "District_enc","Day_enc"
]

X_train = train[FEATURES]
y_train = train["Target"]

X_test = test[FEATURES]

print("Train Features:", X_train.shape)
print("Test Features :", X_test.shape)

Train Features: (875659, 16)
Test Features : (884262, 16)


In [ ]:
# ==========================================
# CELL 9: TRAIN FINAL MODEL
# ==========================================
model = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

model.fit(X_train, y_train)

print("✅ Model Trained")

✅ Model Trained


In [ ]:
# ==========================================
# CELL 10: PREDICT PROBABILITIES
# ==========================================
probs = model.predict_proba(X_test)

print("Prediction Shape:", probs.shape)

Prediction Shape: (884262, 39)


In [ ]:
# ==========================================
# CELL 11: CREATE SUBMISSION FILE
# ==========================================
submission = pd.DataFrame(
    probs,
    columns=le_target.classes_
)

submission.insert(0, "Id", test_raw["Id"])

submission.to_csv("submission.csv", index=False)

print("✅ submission.csv created")
submission.head()

✅ submission.csv created


,Id,ARSON,ASSAULT,BAD CHECKS,BRIBERY,BURGLARY,DISORDERLY CONDUCT,DRIVING UNDER THE INFLUENCE,DRUG/NARCOTIC,DRUNKENNESS,...,SEX OFFENSES NON FORCIBLE,STOLEN PROPERTY,SUICIDE,SUSPICIOUS OCC,TREA,TRESPASS,VANDALISM,VEHICLE THEFT,WARRANTS,WEAPON LAWS
0,0,0.002729,0.149929,3.536676e-44,0.000022,0.027976,0.000544,0.005111,0.015632,0.002853,...,5.094894e-276,0.007740,4.334181e-05,0.054156,0.0,0.005594,0.069682,0.179484,0.028536,0.019650
1,1,0.000697,0.059249,4.043656e-194,0.000005,0.000558,0.000141,0.011895,0.037059,0.001715,...,0.000000e+00,0.002933,3.336876e-184,0.027604,0.0,0.000109,0.033194,0.079306,0.064471,0.018176
2,2,0.003502,0.079538,6.965393e-15,0.000000,0.160099,0.001235,0.000300,0.005712,0.004386,...,0.000000e+00,0.002754,1.903793e-06,0.023695,0.0,0.008225,0.101959,0.091973,0.013108,0.003503
3,3,0.001683,0.119241,9.551427e-19,0.000059,0.016785,0.000955,0.000727,0.016940,0.053321,...,0.000000e+00,0.004154,8.253654e-06,0.037179,0.0,0.003433,0.095748,0.199910,0.028257,0.012494
4,4,0.001683,0.119241,9.551427e-19,0.000059,0.016785,0.000955,0.000727,0.016940,0.053321,...,0.000000e+00,0.004154,8.253654e-06,0.037179,0.0,0.003433,0.095748,0.199910,0.028257,0.012494


In [ ]:
# ==========================================
# CELL 12: DOWNLOAD (COLAB)
# ==========================================
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>